# pLM Covariance Pooling — Run Experiments

**PP1 SoSe2026 | TU München**  
Team: Joel Simon, Julius Schmidt, Lisa Börner, Andreas Weitz

Trains pooler + probe on cached ProtT5 embeddings already in Drive. ProtT5 itself is not loaded here — only the small probe heads (and the PCA / unsupervised poolers) are fit on GPU.

### Workflow
1. GPU check
2. Mount Drive
3. Configuration (Drive paths)
4. Upload + install repo
5. Wire Drive into `data/embeddings/` and `data/raw/`
6. Fit PCA pooler (closed-form, fast)
7. (Optional) Train unsupervised pooler (SGD)
8. Run experiments (all five methods × two tasks)
9. Mirror results back to Drive

## 1 · GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    print('No GPU detected — probe training will be slow on CPU.\n'
          'Runtime -> Change runtime type -> GPU recommended.')
else:
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:  {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Top-level folders in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print(f'  {f}')

## 3 · Configuration

Three things need to be on your Drive **once** — afterwards every Colab session just remounts and goes.

1. **Embeddings** — shortcut to the shared `embeddings/` folder, placed at `MyDrive/embeddings/` (already done).
2. **Labels** — `raw-20260516T183459Z-3-001.zip` at `MyDrive/raw-20260516T183459Z-3-001.zip` (already done).
3. **Code** — `sop_repo.zip` at `MyDrive/sop_repo.zip` (upload once; the cell below falls back to a local picker if it's not there).

Adjust the paths in the next cell if any of those names differ on your Drive.

In [ ]:
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────
EMB_DIR     = Path('/content/drive/MyDrive/embeddings')                       # shared shortcut
LABELS_ZIP  = Path('/content/drive/MyDrive/raw-20260516T183459Z-3-001.zip')   # uploaded once
CODE_ZIP    = Path('/content/drive/MyDrive/sop_repo.zip')                     # uploaded once
RESULTS_OUT = Path('/content/drive/MyDrive/pp1_sop_results')                  # where runs/ lands
# ──────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

checks = {
    'EMB_DIR (folder)':    (EMB_DIR, EMB_DIR.is_dir()),
    'LABELS_ZIP (file)':   (LABELS_ZIP, LABELS_ZIP.is_file()),
    'CODE_ZIP (file)':     (CODE_ZIP, CODE_ZIP.is_file()),
}
print(f'{"Item":<22} {"Status":<10} Path')
print('-' * 90)
for name, (path, ok) in checks.items():
    print(f'{name:<22} {"OK" if ok else "MISSING":<10} {path}')

if EMB_DIR.is_dir():
    found = sorted(EMB_DIR.iterdir())
    print(f'\nEMB_DIR contains {len(found)} files: {[p.name for p in found]}')

## 4 · Install repo

Tries `git clone` first (works if the repo is public). Falls back to `sop_repo.zip` from Drive or a local upload picker.

To build the zip locally: `zip -r sop_repo.zip src scripts configs pyproject.toml README.md tests`

In [ ]:
import subprocess, sys, shutil, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/Julius-Schmidt/pLM-covariance-pooling.git'
REPO_DIR = Path('/content/pLM-covariance-pooling')
LOCAL_ZIP = Path('/content/sop_repo.zip')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Try public git clone first — fast and always up to date.
result = subprocess.run(
    ['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f'Cloned from {REPO_URL}')
else:
    # Fall back to zip (Drive → local → picker).
    if CODE_ZIP.is_file():
        src_zip = CODE_ZIP
        print(f'Using zip from Drive: {CODE_ZIP}')
    elif LOCAL_ZIP.is_file():
        src_zip = LOCAL_ZIP
        print(f'Using local zip: {LOCAL_ZIP}')
    else:
        from google.colab import files
        print('Select sop_repo.zip ...')
        uploaded = files.upload()
        src_zip = Path(f'/content/{next(iter(uploaded))}')

    REPO_DIR.mkdir(parents=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(REPO_DIR)

    # Flatten if the zip added a top-level folder.
    if not (REPO_DIR / 'pyproject.toml').exists():
        inner = next(
            (p for p in REPO_DIR.iterdir() if p.is_dir() and (p / 'pyproject.toml').exists()),
            None,
        )
        if inner is None:
            raise RuntimeError('pyproject.toml not found in zip — check archive layout.')
        for item in inner.iterdir():
            shutil.move(str(item), REPO_DIR / item.name)
        inner.rmdir()
    print(f'Extracted from zip.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=str(REPO_DIR), check=True,
)
print(f'Package installed at {REPO_DIR}')

## 5 · Wire Drive into the repo

Configs use relative paths (`data/embeddings/*.h5`, `data/raw/*/`, `models/`). This cell:
- symlinks `data/embeddings/` → the shared shortcut on Drive,
- extracts the labels zip from Drive into `/content/labels/` and symlinks `data/raw/` → there,
- symlinks `models/` → a folder on Drive so PCA/autoencoder checkpoints survive session resets.

In [ ]:
import shutil, zipfile
from pathlib import Path

DATA_DIR     = REPO_DIR / 'data'
EMB_LINK     = DATA_DIR / 'embeddings'
RAW_LINK     = DATA_DIR / 'raw'
MODELS_LINK  = REPO_DIR / 'models'
MODELS_DRIVE = Path('/content/drive/MyDrive/pp1_sop_models')
LABELS_OUT   = Path('/content/labels')

DATA_DIR.mkdir(exist_ok=True)
MODELS_DRIVE.mkdir(parents=True, exist_ok=True)

def relink(link: Path, target: Path):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

# embeddings → Drive shortcut
relink(EMB_LINK, EMB_DIR)

# models → Drive folder (persists checkpoints across sessions)
relink(MODELS_LINK, MODELS_DRIVE)

# raw → extract labels zip locally, find the deeploc/meltome parent, symlink to it
if LABELS_OUT.exists():
    shutil.rmtree(LABELS_OUT)
LABELS_OUT.mkdir()
with zipfile.ZipFile(LABELS_ZIP) as zf:
    zf.extractall(LABELS_OUT)
raw_root = next(p.parent for p in LABELS_OUT.rglob('deeploc') if p.is_dir())
relink(RAW_LINK, raw_root)

# Verify the four key paths the configs reference
for rel in ['data/embeddings/deeploc_train.h5',
            'data/embeddings/deeploc_test.h5',
            'data/embeddings/meltome_train.h5',
            'data/embeddings/meltome_test.h5',
            'data/raw/deeploc/train_labels.csv',
            'data/raw/deeploc/test_labels.csv',
            'data/raw/meltome/train_labels.csv',
            'data/raw/meltome/test_labels.csv']:
    full = REPO_DIR / rel
    print(f'{rel}: {"OK" if full.exists() else "MISSING"}')

## 5b · Cache embeddings to local SSD

Drive reads are ~200 MB/s; `/content/` is ~2 GB/s. Copying the h5 files once per session (~5–10 min) makes every subsequent read 10× faster — worth it for dc sweeps or multiple runs.

In [ ]:
import shutil, time
from pathlib import Path

LOCAL_EMB = Path('/content/embeddings_local')
LOCAL_EMB.mkdir(exist_ok=True)

H5_FILES = ['deeploc_train.h5', 'deeploc_test.h5', 'meltome_train.h5', 'meltome_test.h5']

for h5 in H5_FILES:
    src = EMB_DIR / h5
    dst = LOCAL_EMB / h5

    # Already cached with correct size → skip.
    if dst.exists() and not dst.is_symlink() and dst.stat().st_size == src.stat().st_size:
        print(f'{h5}: already cached, skipping.')
        continue

    size_bytes = src.stat().st_size
    size_gb = size_bytes / 1e9
    free_bytes = shutil.disk_usage('/content').free

    if free_bytes < size_bytes + 2 * 1024**3:   # keep 2 GB headroom
        # Not enough space — symlink to Drive so the path still resolves.
        if dst.is_symlink():
            dst.unlink()
        elif dst.exists():
            dst.unlink()
        dst.symlink_to(src)
        print(f'{h5}: not enough space ({free_bytes/1e9:.1f} GB free, need {size_gb:.1f} GB) — reading from Drive.')
        continue

    print(f'Copying {h5} ({size_gb:.1f} GB)...', end=' ', flush=True)
    t0 = time.time()
    shutil.copy2(src, dst)
    elapsed = time.time() - t0
    mb_s = size_gb * 1e3 / max(elapsed, 1e-3)
    print(f'done in {elapsed:.0f}s ({mb_s:.0f} MB/s)')

# Repoint the embeddings symlink to LOCAL_EMB (which mixes local files and Drive symlinks).
emb_link = REPO_DIR / 'data' / 'embeddings'
if emb_link.is_symlink():
    emb_link.unlink()
elif emb_link.exists():
    shutil.rmtree(emb_link)
emb_link.symlink_to(LOCAL_EMB)

print(f'\ndata/embeddings/ → {LOCAL_EMB}')
for h5 in H5_FILES:
    p = LOCAL_EMB / h5
    tag = '(Drive symlink)' if p.is_symlink() else '(local SSD)'
    print(f'  {h5}: {tag}')

## 6 · Fit PCA pooler (required for cov_pca configs)

Closed-form, one pass over the embeddings — fast. Fits one checkpoint per dc value referenced by the configs.

In [ ]:
import subprocess, sys

DC_VALUES = [8, 16, 24, 32, 48]   # match the dc sweep grid

def stream(cmd, cwd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=str(cwd))
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {" ".join(cmd)}')

for dc in DC_VALUES:
    out = MODELS_LINK / f'pca_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: exists, skipping.')
        continue
    stream([
        sys.executable, str(REPO_DIR / 'scripts' / 'fit_pca_pool.py'),
        '--embeddings', 'data/embeddings/deeploc_train.h5',
        '--d', '1024', '--dc', str(dc),
        '--output', str(out),
    ], cwd=REPO_DIR)

## 7 · (Optional) Train unsupervised pooler

Frobenius autoencoder via SGD. Needed only for `cov_unsupervised` configs. Skip if you only care about mean / cov_supervised / cov_pca / hybrid.

In [ ]:
for dc in DC_VALUES:
    out = MODELS_LINK / f'unsup_pooler_dc{dc}.pt'
    if out.exists():
        print(f'dc={dc}: exists, skipping.')
        continue
    stream([
        sys.executable, str(REPO_DIR / 'scripts' / 'train_unsupervised_pool.py'),
        '--embeddings', 'data/embeddings/deeploc_train.h5',
        '--d', '1024', '--dc', str(dc),
        '--epochs', '5', '--batch-size', '32', '--lr', '1e-3',
        '--device', DEVICE,
        '--output', str(out),
    ], cwd=REPO_DIR)

## 8 · Run experiments

Pick which configs to run. By default: all five methods on both tasks.

In [ ]:
CONFIGS = [
    'configs/scl/mean.yaml',
    'configs/scl/cov_supervised.yaml',
    'configs/scl/cov_pca.yaml',
    'configs/scl/cov_unsupervised.yaml',
    'configs/scl/hybrid.yaml',
    'configs/meltome/mean.yaml',
    'configs/meltome/cov_supervised.yaml',
    'configs/meltome/cov_pca.yaml',
    'configs/meltome/cov_unsupervised.yaml',
    'configs/meltome/hybrid.yaml',
]

# Single-dc run per config. To sweep instead, set DC_SWEEP = [8, 16, 24, 32, 48].
DC_SWEEP = None

for cfg in CONFIGS:
    print(f'\n=== {cfg} ===')
    cmd = [sys.executable, str(REPO_DIR / 'scripts' / 'run_experiment.py'),
           '--config', cfg]
    if DC_SWEEP:
        cmd += ['--dc'] + [str(d) for d in DC_SWEEP]
    stream(cmd, cwd=REPO_DIR)

## 9 · Mirror results to Drive

In [ ]:
import shutil

src = REPO_DIR / 'results'
if not src.exists():
    print(f'No results dir at {src} — nothing to copy.')
else:
    RESULTS_OUT.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in src.rglob('*'):
        if p.is_file():
            rel = p.relative_to(src)
            dst = RESULTS_OUT / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            n += 1
    print(f'Copied {n} files to {RESULTS_OUT}')

## 10 · Quick look at the summaries

In [ ]:
import json

runs = sorted((REPO_DIR / 'results' / 'runs').glob('*.json'))
for r in runs:
    if r.stem.endswith('_sweep'):
        continue
    d = json.loads(r.read_text())
    metric = 'accuracy_mean' if d['task'] == 'classification' else 'spearman_r_mean'
    std    = metric.replace('_mean', '_std')
    print(f'{d["config"]:<25} {d["method"]:<18} dc={str(d["dc"]):<5} '
          f'{metric.replace("_mean",""):<10}={d[metric]:.4f} ± {d[std]:.4f}')